# 🧪 Production-Grade Model Evaluation & Comparison on AWS Bedrock
### A repeatable, retry-safe harness for choosing and monitoring LLMs in production

This lab builds a complete evaluation pipeline on **Amazon Bedrock**: talk to *any* model through **one unified interface** (the Converse API via LangChain), then **measure, visualize, and cost** how models compare on latency, length, readability, sentiment, token usage, and dollars. Everything here is written to be lifted straight into a real codebase or scheduled job.

---

#### 🎯 Outcomes
1. Connect to Amazon Bedrock reliably across regions, profiles, and IAM roles.
2. Discover — not guess — which models your account is allowed to invoke.
3. Swap models (Amazon Nova, Mistral, Anthropic Claude, …) **without changing parsing code**.
4. Quantify response quality with real metrics and present them as charts.
5. Run a **benchmark harness** with exponential-backoff retries, token tracking, structured logging, and a tidy results table.
6. Estimate token cost and read a multi-dimensional model profile (radar chart).

#### ✅ Prerequisites
- An AWS account with **Amazon Bedrock model access** granted (Console → Bedrock → *Model access*).
- AWS credentials available to this notebook (via a `.env` file, an attached IAM role, or `aws configure`).
- Python 3.10+ (built and validated on 3.12). Runs on **Windows, macOS, and Linux**.

> 💡 **Run top to bottom.** Every analysis section depends on the cells above it. After the install cell, **restart the kernel once** before continuing.

## 🗺️ The big picture — how the pieces fit together

Here is the journey a single prompt takes through the pipeline. Run the cell below to render it.

In [ ]:
# A quick visual of the lab's data flow (pure matplotlib, no internet needed)
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

steps = [
    (".env\n(creds + region)", "#E8F0FE"),
    ("boto3\nSession", "#D2E3FC"),
    ("bedrock-runtime\nclient", "#AECBFA"),
    ("ChatBedrockConverse\n(LangChain)", "#8AB4F8"),
    ("Bedrock model\n(Nova / Mistral / Claude)", "#669DF6"),
    ("Analysis\nmetrics + visuals", "#4285F4"),
]

fig, ax = plt.subplots(figsize=(13, 2.6))
x = 0.0
for i, (label, color) in enumerate(steps):
    ax.add_patch(FancyBboxPatch((x, 0), 1.8, 1.0, boxstyle="round,pad=0.05",
                                linewidth=1.2, edgecolor="#1A1248", facecolor=color))
    ax.text(x + 0.9, 0.5, label, ha="center", va="center", fontsize=9, weight="bold")
    if i < len(steps) - 1:
        ax.annotate("", xy=(x + 2.1, 0.5), xytext=(x + 1.8, 0.5),
                    arrowprops=dict(arrowstyle="->", lw=1.6, color="#1A1248"))
    x += 2.1

ax.set_xlim(-0.2, x); ax.set_ylim(-0.3, 1.4); ax.axis("off")
ax.set_title("Lab data flow: from credentials to comparison", fontsize=13, weight="bold")
plt.tight_layout(); plt.show()

### 🔍 Why this diagram matters
Most Bedrock failures are **not** in the model call itself — they are one box to the left: wrong region, missing credentials, or a model your account can't access. Keep this flow in mind: if a call fails, walk **left to right** and ask *"which box broke?"*

---
## Module 1 — Setup & Environment

**Use case:** Any team that wants to use Bedrock from a notebook, a SageMaker job, or a Lambda needs a reliable, repeatable way to authenticate and pick a region.

**Scenario:** You're a new engineer at a manufacturing analytics startup. IT gave you an AWS account with Bedrock enabled. Your first job is simply to *connect* — no fancy AI yet, just prove the plumbing works.

In [ ]:
# Only UPGRADE the AWS + LangChain stack. Do NOT force-upgrade numpy/scipy/scikit-learn:
# on managed conda/SageMaker images those are pre-built against a specific numpy ABI,
# and a mid-session `-U numpy` is what triggers
# "numpy._core.multiarray failed to import" / "cannot load module more than once".
%pip install -U boto3 botocore
%pip install -U langchain langchain-community langchain-core langchain-aws langsmith

# Add the analysis/plotting libs ONLY if missing — no -U, so the existing,
# ABI-consistent scientific stack on the image is left untouched.
%pip install matplotlib pandas textblob textstat python-dotenv scikit-learn
# ⚠️ After this finishes, RESTART THE KERNEL ONCE, then run from the top.

### 🔍 Why this code
- **`-U` on `boto3`/`botocore` only.** This is the single most common fix for *"model not found / not supported"* errors — the AWS SDK ships a static catalog of model IDs, so an old SDK literally cannot see new models.
- **We deliberately do *not* `-U` numpy / scipy / scikit-learn.** On managed conda images (SageMaker, etc.) these are compiled against a specific numpy ABI. Force-upgrading numpy in a live session breaks that ABI and produces `numpy._core.multiarray failed to import`. We install the analysis libs *without* `-U` so the image's consistent stack is preserved.
- **Restart the kernel once** after this cell. `botocore` loads its API definitions at *import time*, and a swapped-out package can only be cleanly reloaded by a fresh kernel — re-running install cells in the same session is exactly what causes `cannot load module more than once per process`.

> 🧯 **If you still hit a numpy import error:** restart the kernel first. If it persists, realign the stack with `pip install --force-reinstall --no-cache-dir numpy scipy scikit-learn` and restart again.

In [ ]:
# Load credentials/region from a .env file if you have one.
from dotenv import load_dotenv
load_dotenv()   # reads AWS_REGION, AWS_PROFILE, AWS_ACCESS_KEY_ID, etc. if present

In [ ]:
import os
import boto3

# A Session ties together your region + profile so they're used everywhere.
session = boto3.Session(
    region_name=os.getenv("AWS_REGION", "us-east-1"),
    profile_name=os.getenv("AWS_PROFILE"),   # None is fine -> uses default credentials
)

# Two clients, two jobs:
bedrock_runtime = session.client("bedrock-runtime")  # CALL models (invoke / converse)
bedrock         = session.client("bedrock")          # MANAGE: list models, check access

print("Region:", session.region_name)
print("Clients ready:", bool(bedrock_runtime), bool(bedrock))

### 🔍 Why this code
- We build a **`boto3.Session`** first so your `AWS_PROFILE` and `AWS_REGION` are honored. A very common bug is calling `boto3.client(...)` directly — that *silently ignores* your profile.
- Bedrock has **two different clients**, and mixing them up causes confusing errors:
  - `bedrock-runtime` → **uses** models (this is where `converse` / `invoke_model` live).
  - `bedrock` → **manages** models (listing, access, inference profiles).
- We **don't** set `endpoint_url` manually — boto3 derives the correct endpoint from the region. Hard-coding it is a needless source of typos.

> 🪟 **Windows note:** credentials live in `C:\Users\<you>\.aws\credentials`. You can create them with `aws configure` in PowerShell, or set environment variables in a `.env` file next to this notebook.

---
## Module 2 — Connectivity Smoke Test

**Use case:** A health-check / "smoke test" — the cell you run first in CI or after any credential/region change to confirm the path to a model is intact.

**Scenario:** A deployment moved to a new region. Before the pipeline runs, one minimal call confirms credentials, region, and model access all line up — fail fast, with a clear error, instead of deep in the benchmark.

In [ ]:
# The Converse API: ONE request shape that works across every chat model on Bedrock.
prompt = "Hello, world"
model_id = "amazon.nova-lite-v1:0"   # we'll confirm what YOU can access in Module 3

response_check = bedrock_runtime.converse(
    modelId=model_id,
    messages=[{"role": "user", "content": [{"text": prompt}]}],
    inferenceConfig={"maxTokens": 512, "temperature": 0.5},
)

# Converse always returns the text in the same place, no matter the provider:
print(response_check["output"]["message"]["content"][0]["text"])

### 🔍 Why this code (and why **Converse**, not `invoke_model`)
- Bedrock has two calling styles. The older `invoke_model` requires a **different request body and response shape for every provider** (Anthropic ≠ Amazon ≠ Mistral). That's brittle.
- **`converse`** is the unified API: the same `messages=[{"role": ..., "content": [{"text": ...}]}]` works everywhere, and the reply is *always* at `output → message → content[0] → text`.
- `inferenceConfig` is where standardized knobs live: `maxTokens`, `temperature`, `topP`.

### 🧩 If this cell errors, read the error type
| Error | Meaning | Fix |
|---|---|---|
| `AccessDeniedException` | Model not enabled for your account | Enable it in Bedrock → *Model access* |
| `ValidationException: ... not support` | You picked an embedding/image model | Use a **text/chat** model |
| `ResourceNotFoundException` | Wrong ID, or legacy/retired model | Use Module 3 to find a valid one |

---
## Module 3 — Discover What Your Account Can Use

**Use case:** Stop guessing model IDs. Ask AWS directly which models exist in your region and which are `ACTIVE`.

**Scenario:** Your teammate's notebook used `claude-3-sonnet`, but it now errors as *Legacy*. Instead of trial-and-error, you query the catalog and pick from what's truly available **to you**.

In [ ]:
from botocore.exceptions import ClientError

# 1) Text models available on-demand in THIS region
resp = bedrock.list_foundation_models(byOutputModality="TEXT", byInferenceType="ON_DEMAND")

print("On-demand TEXT models in", session.region_name, "\n" + "-" * 70)
for m in resp["modelSummaries"]:
    status = m.get("modelLifecycle", {}).get("status", "?")   # ACTIVE / LEGACY
    print(f'{m["modelId"]:60} {status}')

# 2) Inference profiles (the us./eu./jp. IDs) — optional, needs an extra IAM permission
print("\nInference profiles\n" + "-" * 70)
if hasattr(bedrock, "list_inference_profiles"):
    try:
        for p in bedrock.list_inference_profiles().get("inferenceProfileSummaries", []):
            print(p["inferenceProfileId"], "->", p["status"])
    except ClientError as e:
        print(f"(Skipped: {e.response['Error']['Code']} — your role lacks "
              f"bedrock:ListInferenceProfiles. This is optional, continue anyway.)")
else:
    print("(Upgrade boto3 to list inference profiles, then restart the kernel.)")

### 🔍 Why this code
- `list_foundation_models` is the **source of truth** for *"what can I actually call?"* — filtered here to on-demand **text** models.
- `modelLifecycle.status` tells you `ACTIVE` vs `LEGACY`. **Never** pick a `LEGACY` model: AWS blocks it if you haven't used it recently.
- The inference-profile listing is wrapped in `try/except` because it needs a separate IAM permission (`bedrock:ListInferenceProfiles`). It's a *nice-to-have*, so we let the cell finish even if your role can't call it.

> 📝 **Write down two `ACTIVE` model IDs** from the output — you'll plug them into Module 5. Common safe picks: `amazon.nova-lite-v1:0`, `amazon.nova-pro-v1:0`, `mistral.mistral-small-2402-v1:0`.

---
## Module 4 — One Interface for Every Model (LangChain)

**Use case:** You want to A/B test models, or let your app fall back from an expensive model to a cheaper one — without rewriting parsing logic each time.

**Scenario:** Product asks: *"Can we switch from Model A to Model B next week?"* If your code is tied to one provider's response format, that's a painful rewrite. We make the swap a one-line change.

In [ ]:
import os
import boto3
from langchain_aws import ChatBedrockConverse


def get_llm(model_kwargs, model_id="amazon.nova-lite-v1:0"):
    """Return a LangChain chat model backed by Amazon Bedrock (Converse API).

    Parameters
    ----------
    model_kwargs : dict   e.g. {"temperature": 0.1, "max_tokens": 4096}
    model_id     : str    a Bedrock model ID or inference-profile ID
    """
    session = boto3.Session(
        region_name=os.getenv("AWS_REGION", "us-east-1"),
        profile_name=os.getenv("AWS_PROFILE"),
    )
    client = session.client("bedrock-runtime")
    return ChatBedrockConverse(client=client, model=model_id, **model_kwargs)


def to_text(x):
    """Pull plain text out of ANY response shape this lab produces."""
    if isinstance(x, str):                       # already a string
        return x
    if isinstance(x, dict):                      # raw Converse API response
        return x["output"]["message"]["content"][0]["text"]
    content = x.content                          # LangChain AIMessage
    if isinstance(content, list):                # content blocks (e.g. thinking on)
        return "".join(b.get("text", "") for b in content if isinstance(b, dict))
    return content


print("get_llm() and to_text() are ready.")

### 🔍 Why this code
- **`ChatBedrockConverse`** is the modern, recommended LangChain class. It runs on the Converse API, so the *same code* drives Nova, Mistral, Claude, Llama, etc. (The old `BedrockChat` from `langchain_community` is deprecated.)
- **`get_llm`** is a tiny factory: pass a model ID + params, get back a ready-to-use model object. Swapping models = changing one string.
- **`to_text` is the unsung hero.** Different things return different shapes:
  - a raw `converse` call → a **dict**
  - `llm.invoke(...)` → an **AIMessage** whose `.content` is usually a string…
  - …but can be a **list of content blocks** for reasoning/"thinking" models.
  `to_text` handles all three, so the rest of the lab never crashes on `'dict' has no attribute content` or `'list' has no attribute split`.

---
## Module 5 — Choose Two Models & Generate Responses

**Use case:** Run the *identical* prompt through two models so any difference you measure later is about the **models**, not the question.

**Scenario:** Your manufacturing client asks an LLM for industry trends. You want to know: which model is faster? More detailed? Easier to read? First, collect the raw material — two responses to one prompt.

In [ ]:
# 👉 EDIT THESE to two IDs that showed ACTIVE in Module 3.
# Add a us./eu./jp. prefix if a model needs an inference profile,
# e.g. "us.anthropic.claude-opus-4-8".
MODEL_1, NAME_1 = "amazon.nova-lite-v1:0", "Amazon Nova Lite"
MODEL_2, NAME_2 = "amazon.nova-pro-v1:0",  "Amazon Nova Pro"
# Example alternative that may be active on your account:
#   MODEL_2, NAME_2 = "mistral.mistral-small-2402-v1:0", "Mistral Small"

model_kwargs = {"max_tokens": 4096, "temperature": 0.1}

llm   = get_llm(model_id=MODEL_1, model_kwargs=model_kwargs)
llm_2 = get_llm(model_id=MODEL_2, model_kwargs=model_kwargs)
print("Comparing:", NAME_1, "vs", NAME_2)

In [ ]:
import time

prompt = "Tell me about the top 3 trends in Industrial Manufacturing"

t0 = time.time(); response  = llm.invoke(prompt);   execution_time  = time.time() - t0
t1 = time.time(); response2 = llm_2.invoke(prompt);  execution_time2 = time.time() - t1

print(f"=== {NAME_1}  ({execution_time:.2f}s) ===\n{to_text(response)}\n")
print(f"=== {NAME_2}  ({execution_time2:.2f}s) ===\n{to_text(response2)}")

### 🔍 Why this code
- We define the models as **named variables** (`NAME_1`, `NAME_2`) so every chart label downstream stays correct automatically — change the model, and the labels follow. No more hard-coded "Claude 2.1" captions that lie about what actually ran.
- `temperature=0.1` keeps answers focused and **repeatable**, which matters when you're comparing — high randomness would muddy the comparison.
- We wrap each call in `time.time()` to capture **latency**, our first metric.
- Both responses are saved (`response`, `response2`) because *every* analysis cell below reuses them. Generating once and reusing avoids paying for the same call repeatedly.

---
## Module 6 — Measuring Response Quality

**Use case:** "Better" is vague. Turn it into numbers: how **similar** are the answers, how **long**, how **readable**, how **positive** in tone?

**Scenario:** Stakeholders ask *"are these two models basically saying the same thing?"* You answer with a similarity score and a clean metrics table instead of a hunch.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from textblob import TextBlob
from textstat import flesch_reading_ease
import pandas as pd

text1, text2 = to_text(response), to_text(response2)

# Similarity describes the PAIR of answers (0 = totally different, 1 = identical)
tfidf = TfidfVectorizer().fit_transform([text1, text2])
similarity = float(cosine_similarity(tfidf[0:1], tfidf[1:2])[0][0])

# Per-model metrics, collected into a tidy table
metrics = pd.DataFrame([
    {"model": NAME_1, "latency_s": round(execution_time, 2),  "words": len(text1.split()),
     "readability": round(flesch_reading_ease(text1), 1),
     "sentiment": round(TextBlob(text1).sentiment.polarity, 3)},
    {"model": NAME_2, "latency_s": round(execution_time2, 2), "words": len(text2.split()),
     "readability": round(flesch_reading_ease(text2), 1),
     "sentiment": round(TextBlob(text2).sentiment.polarity, 3)},
]).set_index("model")

print(f"Response similarity (0=different, 1=identical): {similarity:.2f}\n")
metrics   # rendered as a table by Jupyter

### 🔍 Why this code & what each metric means
- **Similarity (TF-IDF + cosine):** turns each answer into a word-importance vector, then measures the angle between them. ~`0.8+` means the models largely agree; a low score means they took different angles. It's a *pair* metric, so it's printed once, not per model.
- **Latency (s):** wall-clock time per call — your speed/UX signal.
- **Words:** crude length/verbosity. More isn't always better; pair it with readability.
- **Readability (Flesch Reading Ease):** higher = easier to read (60–70 ≈ plain English; below 30 ≈ dense/academic).
- **Sentiment polarity:** −1 (negative) → +1 (positive). For factual content this usually sits near 0; useful for catching a model that's oddly upbeat or negative.
- We put everything in a **pandas DataFrame** so the next cell can chart it directly — and so Jupyter renders a clean table for free.

---
## Module 7 — A Comparison Dashboard

**Use case:** One glance should tell a non-technical stakeholder which model wins on which dimension.

**Scenario:** You're presenting to the client. A wall of numbers won't land — a four-panel dashboard will.

In [ ]:
import matplotlib.pyplot as plt

panels = [
    ("latency_s",   "Execution time (s) — lower is faster"),
    ("words",       "Response length (words)"),
    ("readability", "Readability (higher = easier)"),
    ("sentiment",   "Sentiment polarity (-1 to +1)"),
]
bar_colors = ["#4285F4", "#34A853"]

fig, axes = plt.subplots(2, 2, figsize=(11, 7))
fig.suptitle(f"{NAME_1}  vs  {NAME_2}", fontsize=14, weight="bold")

for ax, (col, title) in zip(axes.ravel(), panels):
    bars = ax.bar(metrics.index, metrics[col], color=bar_colors)
    ax.set_title(title, fontsize=10)
    ax.margins(y=0.18)
    for bar, val in zip(bars, metrics[col]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                f"{val}", ha="center", va="bottom", fontsize=9)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

### 🔍 Why this code
- A **2×2 subplot grid** (`plt.subplots(2, 2)`) shows four metrics in one figure — far easier to absorb than four separate charts.
- We **annotate each bar** with its value (`ax.text(...)`) so the chart is readable without hunting at the axis.
- `ax.margins(y=0.18)` adds headroom so the value labels don't get clipped at the top.
- Because the data comes straight from the `metrics` DataFrame and labels come from `NAME_1/NAME_2`, the dashboard updates automatically when you change models — nothing to hand-edit.

---
## Module 8 — A Production Benchmark Harness

**Use case:** Real evaluation isn't one prompt — it's *many* prompts across *several* models, run reliably even when the API throttles you, with token usage captured for cost analysis.

**Scenario:** Leadership wants a defensible recommendation: *"Which model should we standardize on?"* You need repeatable, retry-safe measurements you can re-run any time and hand to finance.

In [ ]:
import time, logging
import pandas as pd
from botocore.exceptions import ClientError

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger("bedrock-lab")


def invoke_with_retry(llm, prompt, max_retries=4, base_delay=2):
    """Invoke a model, retrying with exponential backoff on throttling."""
    for attempt in range(max_retries):
        try:
            return llm.invoke(prompt)
        except ClientError as e:
            code = e.response["Error"]["Code"]
            transient = code in ("ThrottlingException", "TooManyRequestsException",
                                 "ServiceUnavailableException", "ModelTimeoutException")
            if transient and attempt < max_retries - 1:
                delay = base_delay * (2 ** attempt)
                log.warning(f"{code}: retrying in {delay}s "
                            f"(attempt {attempt + 1}/{max_retries})")
                time.sleep(delay)
                continue
            raise   # not transient, or out of retries -> surface the real error


def usage_of(msg):
    """Token counts from a LangChain AIMessage, if the provider reported them."""
    u = getattr(msg, "usage_metadata", None) or {}
    return u.get("input_tokens"), u.get("output_tokens"), u.get("total_tokens")


def benchmark(models: dict, prompts: list, model_kwargs=None) -> pd.DataFrame:
    """Run every prompt through every model; return a tidy results DataFrame."""
    model_kwargs = model_kwargs or {"max_tokens": 1024, "temperature": 0.2}
    rows = []
    for name, model_id in models.items():
        llm = get_llm(model_id=model_id, model_kwargs=model_kwargs)
        for prompt in prompts:
            t0 = time.time()
            msg = invoke_with_retry(llm, prompt)
            latency = time.time() - t0
            text = to_text(msg)
            inp, out, tot = usage_of(msg)
            log.info(f"{name:18} {latency:5.2f}s  {len(text.split()):4d} words")
            rows.append({
                "model": name,
                "prompt": (prompt[:38] + "…") if len(prompt) > 38 else prompt,
                "latency_s": round(latency, 2),
                "words": len(text.split()),
                "input_tokens": inp, "output_tokens": out, "total_tokens": tot,
                "text": text,
            })
    return pd.DataFrame(rows)


print("Production harness ready: invoke_with_retry(), usage_of(), benchmark()")

### 🔍 Why this code (this is what separates a demo from production)
- **`invoke_with_retry` with exponential backoff:** Bedrock *will* throttle you under load (`ThrottlingException`). Naively, that crashes your run. We retry with growing delays (2s, 4s, 8s…) for *transient* errors only — and immediately re-raise real errors (like `AccessDenied`) so you don't mask genuine problems.
- **`usage_of` captures token usage** from `usage_metadata`. Tokens — not wall-clock time — drive your **bill**, so we record them. We use `.get(...)` because some providers don't always report every field.
- **`benchmark` returns a tidy DataFrame** (one row per model×prompt). Tidy data is the key to easy charts, aggregation, and export to CSV/Excel for stakeholders.
- **Logging instead of `print`** gives timestamps and levels — the difference between "it's running" and an auditable record you can paste into a ticket.

---
## Module 9 — Visualizing the Benchmark

**Use case:** Convert the raw benchmark table into the two visuals decision-makers actually use: a **per-prompt latency comparison** and a **normalized model profile** (radar).

**Scenario:** Two models are close on average, but one is consistently slower on long prompts. A grouped bar chart exposes that; a radar chart summarizes the overall trade-off in a single shape.

In [ ]:
# Run the benchmark across both models and a small prompt suite
models = {NAME_1: MODEL_1, NAME_2: MODEL_2}
bench_prompts = [
    "Explain automation and robotics in manufacturing.",
    "Explain digital transformation and Industry 4.0.",
    "Explain sustainability in industrial operations.",
]

df = benchmark(models, bench_prompts, model_kwargs={"max_tokens": 512, "temperature": 0.2})
df[["model", "prompt", "latency_s", "words", "output_tokens"]]

In [ ]:
# VISUAL 1: latency per prompt, grouped by model
pivot = df.pivot_table(index="prompt", columns="model", values="latency_s")
ax = pivot.plot(kind="bar", figsize=(10, 5), color=["#4285F4", "#34A853"], edgecolor="white")
ax.set_ylabel("Latency (seconds)")
ax.set_title("Latency by prompt and model — lower is better")
ax.legend(title="Model")
plt.xticks(rotation=18, ha="right")
plt.tight_layout(); plt.show()

In [ ]:
# VISUAL 2: normalized radar/spider profile (outer edge = better)
import numpy as np
from textstat import flesch_reading_ease
from textblob import TextBlob

agg = df.groupby("model").agg(latency=("latency_s", "mean"),
                              words=("words", "mean")).copy()
text_by_model = df.groupby("model")["text"].apply(" ".join)
agg["readability"] = text_by_model.apply(flesch_reading_ease)
agg["sentiment"]   = text_by_model.apply(lambda t: TextBlob(t).sentiment.polarity)
agg["speed"]       = 1.0 / agg["latency"]          # invert: faster -> higher = better

radar_cols = ["speed", "words", "readability", "sentiment"]
norm = agg[radar_cols].copy()
for c in radar_cols:                                # min-max normalize each axis to 0..1
    lo, hi = norm[c].min(), norm[c].max()
    norm[c] = 0.5 if hi == lo else (norm[c] - lo) / (hi - lo)

labels = ["Speed", "Detail", "Readability", "Positivity"]
angles = np.linspace(0, 2 * np.pi, len(labels), endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(6.5, 6.5), subplot_kw=dict(polar=True))
for (model, row), color in zip(norm.iterrows(), ["#4285F4", "#34A853"]):
    vals = row.tolist() + [row.tolist()[0]]
    ax.plot(angles, vals, color=color, linewidth=2, label=model)
    ax.fill(angles, vals, color=color, alpha=0.15)
ax.set_xticks(angles[:-1]); ax.set_xticklabels(labels, fontsize=10)
ax.set_yticklabels([]); ax.set_ylim(0, 1)
ax.set_title("Normalized model profile (outer = better)", fontsize=12, weight="bold", pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.25, 1.1))
plt.tight_layout(); plt.show()

### 🔍 Why this code
- **`pivot_table` + `.plot(kind="bar")`** is the fastest path from tidy data to a grouped bar chart — pandas handles the grouping; you just label it.
- The **radar chart** compares several dimensions at once. Because the metrics have different units (seconds vs words vs a 0–100 score), we **min-max normalize** each axis to 0–1 and **invert latency into "speed"** so that on every axis *bigger = better*. The model with the larger enclosed area is the more balanced all-rounder.
- We compute readability/sentiment on the **concatenated** text per model so the profile reflects the model's overall style across the whole prompt suite, not a single answer.

---
## Module 10 — Prompt Engineering Experiments

**Use case:** The *prompt* often matters more than the model. Compare zero-shot vs few-shot vs multilingual vs long-form on the same model and see the effect on length and latency.

**Scenario:** Before paying for a bigger model, you test whether smarter prompting on the current one already gets you there.

In [ ]:
few_shot_prompt = """
Example 1: What are the benefits of AI? -> AI enables automation and improves efficiency.
Example 2: Explain cloud computing.   -> Cloud computing delivers computing services over the internet.
Now, explain industrial manufacturing trends.
"""

experiments = {
    "Zero-shot":          "Explain industrial manufacturing trends.",
    "Few-shot":           few_shot_prompt,
    "Multilingual (FR)":  "Expliquez les tendances actuelles dans la fabrication industrielle.",
    "Long-form report":   "Write a detailed report on the impact of robotics in industrial manufacturing.",
}

exp_rows = []
for label, p in experiments.items():
    t0 = time.time()
    msg = invoke_with_retry(llm, p)
    elapsed = time.time() - t0
    txt = to_text(msg)
    exp_rows.append({
        "experiment": label,
        "latency_s": round(elapsed, 2),
        "words": len(txt.split()),
        "preview": txt[:110].replace("\n", " ") + "…",
    })

exp_df = pd.DataFrame(exp_rows).set_index("experiment")
exp_df

In [ ]:
# Visual: how prompt style changes response length
ax = exp_df["words"].plot(kind="barh", figsize=(9, 4), color="#FBBC04", edgecolor="white")
ax.set_xlabel("Response length (words)")
ax.set_title(f"Prompt style vs response length — {NAME_1}")
for i, v in enumerate(exp_df["words"]):
    ax.text(v, i, f" {v}", va="center", fontsize=9)
plt.tight_layout(); plt.show()

### 🔍 Why this code
- We reuse the **same model** (`llm`) and only change the **prompt style**, isolating the prompt's effect — a clean controlled experiment.
- **Few-shot** prompting (showing examples) steers format and depth; **multilingual** confirms the model answers in the input language; **long-form** stress-tests verbosity and latency.
- Results land in a DataFrame with a short **`preview`** column so you can eyeball quality without scrolling through full outputs.
- The horizontal bar chart makes the length differences obvious at a glance — often the cheapest win is a better prompt, not a bigger model.

---
## Module 11 — Token Efficiency & Cost Estimate

**Use case:** Latency is UX; **tokens are money**. Translate your benchmark into an estimated dollar cost so the model choice can be justified to finance.

**Scenario:** Two models perform similarly, but one costs 5× more per token. This cell turns the recommendation into a budget line item.

In [ ]:
# ⚠️ PLACEHOLDER PRICES — pricing changes and varies by region/model.
# Look up the CURRENT numbers on the AWS Bedrock pricing page and edit these.
# Units: USD per 1,000,000 tokens.
PRICING = {
    NAME_1: {"input": 0.06, "output": 0.24},
    NAME_2: {"input": 0.80, "output": 3.20},
}

cost = df.groupby("model").agg(
    input_tokens=("input_tokens", "sum"),
    output_tokens=("output_tokens", "sum"),
).fillna(0)

def est_cost(row):
    p = PRICING.get(row.name)
    if not p:
        return None
    return round(row["input_tokens"] / 1e6 * p["input"]
                 + row["output_tokens"] / 1e6 * p["output"], 6)

cost["est_cost_usd"] = cost.apply(est_cost, axis=1)
cost

### 🔍 Why this code
- We aggregate **token usage from the benchmark** (`df`), so cost is grounded in real measured calls, not guesses.
- Pricing is split into **input vs output** because output tokens almost always cost more — and chatty models quietly run up the output bill.
- `PRICING` is a clearly-flagged, **editable placeholder**. Always verify against the live AWS pricing page; never ship estimates built on hard-coded numbers you haven't checked.
- If `est_cost_usd` is `0` or `None`, your provider didn't return token counts for those calls — note that some models omit `usage_metadata`, in which case estimate from word count (≈ 0.75 words/token) as a rough fallback.

---
## ✅ Wrap-up, Cleanup & Troubleshooting

**What you built:** a full evaluation pipeline — connect → discover → unify → measure → benchmark → visualize → cost — that works for *any* Bedrock chat model by changing two strings in Module 5.

### 🧹 Cleanup
Amazon Bedrock is **serverless** and on-demand: there are **no clusters or endpoints to delete**, so this lab leaves nothing running to be charged for. (If you later create *Provisioned Throughput* or a *Knowledge Base*, those **do** incur cost and must be deleted manually.)

### 🧯 Troubleshooting quick-reference (every error we designed around)
| Symptom | Root cause | Fix |
|---|---|---|
| `SyntaxError: ':' expected` | Model ID placed inside the request body | Model ID goes in `modelId=`, not the body |
| `KeyError: 'content'` | Parsing an Anthropic shape from a non-Anthropic model | Use `converse` + `to_text()` |
| `ValidationException: ... not support` | Picked an embedding/image model | Use a **text/chat** model |
| `ResourceNotFoundException: Legacy` | Retired model ID | Pick an `ACTIVE` model (Module 3) |
| `AccessDeniedException: not available` | Model not enabled for the account | Bedrock Console → *Model access* |
| `AccessDeniedException: ListInferenceProfiles` | IAM role missing the action | Add `bedrock:ListInferenceProfiles` (optional) |
| `AttributeError: no attribute 'list_inference_profiles'` | Old boto3 | `pip install -U boto3 botocore` + **restart kernel** |
| `numpy._core.multiarray failed to import` | numpy ABI broken by a mid-session `-U` upgrade | **Restart kernel**; if needed `pip install --force-reinstall numpy scipy scikit-learn` |
| `'dict'/'list' object has no attribute ...` | Mixed response shapes | Always parse via `to_text()` |
| `NameError: response2` | Variable overwritten/reused | Generate both responses in one cell |

### 🚀 Next steps
- Export results for stakeholders: `df.to_csv("benchmark.csv")` or `df.to_excel("benchmark.xlsx")`.
- Add **streaming** (`ChatBedrockConverse(..., streaming=True)`) to measure *time-to-first-token*, the real UX metric.
- Add **tool/function calling** and re-run the benchmark on agentic tasks.
- Wire the benchmark into a scheduled job so model quality is tracked over time.